# House Prices — Ensemble (XGBoost + LightGBM + CatBoost) (Colab)

Kaggle [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) 대회를 **부스팅 3대장(XGBoost · LightGBM · CatBoost)** 으로 각각 학습한 뒤, 예측을 **평균 블렌딩(blending)** 하는 앙상블 예제입니다.

- 서로 다른 모델의 예측을 평균하면 보통 단일 모델보다 안정적이고 정확해집니다.
- 실행 시 **Kaggle API로 데이터를 직접 다운로드**합니다 (`train.csv`/`test.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 마지막에 제출 파일 `submission.csv` 가 `/content` 에 생성됩니다.

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "xgboost", "lightgbm", "catboost", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 House Prices 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 준비 (공통 전처리)

세 모델이 같은 입력을 쓰도록 통일된 전처리를 적용합니다: 범주형은 **Ordinal 인코딩**, 숫자형 결측치는 **중앙값**으로 채웁니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
print("train:", train.shape, "| test:", test.shape)

y = train["SalePrice"]
X = train.drop(["Id", "SalePrice"], axis=1)
X_test = test.drop(["Id"], axis=1)

# 범주형 / 숫자형 컬럼 구분 (최신 pandas 호환: string dtype 포함)
cat_cols = list(X.select_dtypes(include=["object", "string", "category"]).columns)
num_cols = [c for c in X.columns if c not in cat_cols]
print("범주형:", len(cat_cols), "| 숫자형:", len(num_cols))

# 범주형: 결측 채우고 Ordinal 인코딩
for c in cat_cols:
    X[c] = X[c].fillna("NA").astype(str)
    X_test[c] = X_test[c].fillna("NA").astype(str)
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

# 숫자형: 중앙값으로 결측 채움
for c in num_cols:
    med = X[c].median()
    X[c] = X[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print("train/val:", X_tr.shape, X_val.shape)

## 3. 모델 1 — XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=2000, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    early_stopping_rounds=50,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

pred_xgb_val = xgb.predict(X_val)
pred_xgb_test = xgb.predict(X_test)
print("XGBoost done | best_iteration:", xgb.best_iteration)

## 4. 모델 2 — LightGBM

In [ ]:
from lightgbm import LGBMRegressor
import lightgbm as lgb

lgbm = LGBMRegressor(
    n_estimators=2000, learning_rate=0.05, num_leaves=31,
    subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1,
)
lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
         callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

pred_lgb_val = lgbm.predict(X_val)
pred_lgb_test = lgbm.predict(X_test)
print("LightGBM done | best_iteration:", lgbm.best_iteration_)

## 5. 모델 3 — CatBoost

In [ ]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=2000, learning_rate=0.05, depth=6, loss_function="RMSE",
    random_seed=42, early_stopping_rounds=50, verbose=0,
)
cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

pred_cat_val = cat.predict(X_val)
pred_cat_test = cat.predict(X_test)
print("CatBoost done | best_iteration:", cat.get_best_iteration())

## 6. 앙상블 (평균 블렌딩) & 비교

세 모델의 검증 MAE 와, 예측을 평균한 앙상블의 MAE 를 비교합니다. 보통 앙상블이 가장 낮은 오차를 보입니다.

In [ ]:
from sklearn.metrics import mean_absolute_error

blend_val = (pred_xgb_val + pred_lgb_val + pred_cat_val) / 3
blend_test = (pred_xgb_test + pred_lgb_test + pred_cat_test) / 3

for name, pv in [("XGBoost", pred_xgb_val), ("LightGBM", pred_lgb_val),
                 ("CatBoost", pred_cat_val), ("Ensemble", blend_val)]:
    print(f"{name:10s} Val MAE = {mean_absolute_error(y_val, pv):,.1f}")

## 7. 제출 파일 생성

In [ ]:
submission_path = os.path.join(WORK_DIR, "submission.csv")
output = pd.DataFrame({"Id": test["Id"], "SalePrice": blend_test})
output.to_csv(submission_path, index=False)
print("Your submission was successfully saved to:", submission_path)
output.head()

## 8. 제출 파일 내려받기

`submission.csv` 가 `/content` 에 생성되었습니다. 좌측 파일창에서 받거나 아래 셀로 바로 내려받으세요.

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[House Prices 제출 페이지](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/submit)**

- 대회 홈: https://www.kaggle.com/c/house-prices-advanced-regression-techniques
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.